In [43]:
import os
import getpass
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,AIMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessageGraph,StateGraph
from langchain_groq import chat_models,ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnableConfig

In [2]:
load_dotenv()

True

In [3]:
os.environ["LANGSMITH_TRACING"] = "true"
if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

In [4]:
print("GROQ_API_KEY:", os.getenv("GROQ_API_KEY"))
print("LANGSMITH_API_KEY:", os.getenv("LANGSMITH_API_KEY"))
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT"))
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))
print("LANGSMITH_ENDPOINT:", os.getenv("LANGSMITH_ENDPOINT"))

GROQ_API_KEY: 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
LANGSMITH_API_KEY: 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
LANGSMITH_PROJECT: RAG_chatbot
LANGSMITH_TRACING: true
LANGSMITH_ENDPOINT: https://api.smith.langchain.com


In [5]:
model = ChatGroq(groq_api_key=os.getenv("GROQ_API_KEY"),model_name="llama3-8b-8192")

In [6]:
response = model.invoke("How are you ?")

In [7]:
print(response)

content="I'm just an AI, I don't have emotions or feelings like humans do. I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 14, 'total_tokens': 57, 'completion_time': 0.035833333, 'prompt_time': 0.0029815, 'queue_time': 0.233147831, 'total_time': 0.038814833}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_a97cfe35ae', 'finish_reason': 'stop', 'logprobs': None} id='run-6a748ecd-950c-4544-ad2f-b7b22e326480-0' usage_metadata={'input_tokens': 14, 'output_tokens': 43, 'total_tokens': 57}


In [8]:
model.invoke([HumanMessage(content="Hi, I'm Vivek Soni,How about you ?")])

AIMessage(content="Nice to meet you, Vivek Soni! I'm LLaMA, an AI assistant developed by Meta AI that can understand and respond to human input in a conversational manner. I don't have a personal identity or feelings, but I'm here to assist you with any questions or topics you'd like to discuss! What brings you here today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 23, 'total_tokens': 95, 'completion_time': 0.06, 'prompt_time': 0.003316027, 'queue_time': 0.239405498, 'total_time': 0.063316027}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_a97cfe35ae', 'finish_reason': 'stop', 'logprobs': None}, id='run-dbb5612d-b946-4a0a-8106-6fe714178fce-0', usage_metadata={'input_tokens': 23, 'output_tokens': 72, 'total_tokens': 95})

## keep conversation history along with the query to get proper output.

In [9]:
# from langchain_core.messages import AIMessage

# llm_response = model.invoke(
#     [
#         HumanMessage(content="Hi, I'm Vivek Soni,How about you ?"),
#         AIMessage(content="Nice to meet you, Vivek Soni!"),
#         HumanMessage(content="What's my name?")
#     ]
# )

In [10]:
# output_parser = StrOutputParser()
# output_parser.invoke(llm_response)

## Chains 

In [11]:
# chain = model | output_parser
# chain.invoke("Hello, I'm Vivek Soni!, how are you ?")

In [12]:
# template = ChatPromptTemplate([
#     'System','You are a Trading and investment expert. ',
#     'human','I\'n having {Question}'
# ])

In [13]:
# template.invoke({"Question":["what is intraday trading ?"]})

In [14]:
# chain = template | model | output_parser
# chain.invoke({"Question":["what is intraday trading ?"]})

### Processing PDFs

In [15]:
import os
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    DirectoryLoader,
)

from langchain.text_splitter import RecursiveCharacterTextSplitter


pdf_directory = "../SourceFile/"

pdf_files = [
    os.path.join(pdf_directory, f)
    for f in os.listdir(pdf_directory)
    if f.endswith(".pdf")
]

docs = []

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

for pdf_file in pdf_files:
    print(f"Processing: {pdf_file}")

    loader = PyPDFLoader(pdf_file)
    documents = loader.load()

    split_docs = text_splitter.split_documents(documents)
    docs.extend(split_docs)

print(f"Total chunks created: {len(docs)}")
print(f"Sample chunk:\n{docs[0].page_content}" if docs else "No documents found.")

Processing: ../SourceFile/JSCO21_IISem_StockMarketOperations.pdf


Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 149 0 (offset 0)
Ignoring wrong pointing object 299 0 (offset 0)
Ignoring wrong pointing object 489 0 (offset 0)
Ignoring wrong pointing object 491 0 (offset 0)


Processing: ../SourceFile/Stock-Investing-101-eBook.pdf
Processing: ../SourceFile/TA_wrkbk.pdf
Processing: ../SourceFile/The-Complete-Guide-to-Trading.pdf
Total chunks created: 2518
Sample chunk:
DIRECTORATE OF DISTANCE 
& 
CONTINUING EDUCATION 
B.COM II Semester 2023-24 
Stock Market Operation 
 
EDITED BY 
Dr.M.NITHYA M.COM.,M.Phil.,M.B.A.,TN-SET.,Ph.D 
ASSISTANT PROFESSOR (T) 
DEPARTMENT OF COMMERCE 
MANONMANIAM SUNDARANAR UNIVERISTY 
TIRUNELVELI - 12


### Extracticing data from wikipedia

In [16]:
from langchain_community.document_loaders import WikipediaLoader

wiki_documents = []

txt_directory = "../SourceFile/"
txt_files = [os.path.join(txt_directory, f) for f in os.listdir(txt_directory) if f.endswith(".txt")]

for file in txt_files:
    file_open = open(file,"r",encoding="utf-8")
    for url in file_open:
        url = url.strip()
        title = url.split("/")[-1]  
        loader = WikipediaLoader(query=title, load_max_docs=1)
        documents = loader.load()
        wiki_doc = text_splitter.split_documents(documents)
        wiki_documents.extend(wiki_doc)

In [17]:
wiki_documents[0].page_content

'A stock market, equity market, or share market is the aggregation of buyers and sellers of stocks (also called shares), which represent ownership claims on businesses; these may include securities listed on a public stock exchange as well as stock that is only traded privately, such as shares of private companies that are sold to investors through equity crowdfunding platforms. Investments are usually made with an investment strategy in mind.'

In [18]:
new_doc = docs + wiki_documents
new_doc

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-03-07T11:57:24+00:00', 'source': '../SourceFile/JSCO21_IISem_StockMarketOperations.pdf', 'total_pages': 219, 'page': 0, 'page_label': '1'}, page_content='DIRECTORATE OF DISTANCE \n& \nCONTINUING EDUCATION \nB.COM II Semester 2023-24 \nStock Market Operation \n \nEDITED BY \nDr.M.NITHYA M.COM.,M.Phil.,M.B.A.,TN-SET.,Ph.D \nASSISTANT PROFESSOR (T) \nDEPARTMENT OF COMMERCE \nMANONMANIAM SUNDARANAR UNIVERISTY \nTIRUNELVELI - 12'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-03-07T11:57:24+00:00', 'source': '../SourceFile/JSCO21_IISem_StockMarketOperations.pdf', 'total_pages': 219, 'page': 1, 'page_label': '2'}, page_content='STOCK MARKET OPERATIONS \nSubject \nCode \n \nL \n \nT \n \nP \n \nS \n \nCredits Inst. \nHours \nMarks \nCIA External Total \n     2 2 25 75 100 \nLearning Objectives: \nLO1: To acquaint students with knowledge o

In [19]:
len(new_doc)

2594

In [33]:
docs

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-03-07T11:57:24+00:00', 'source': '../SourceFile/JSCO21_IISem_StockMarketOperations.pdf', 'total_pages': 219, 'page': 0, 'page_label': '1'}, page_content='DIRECTORATE OF DISTANCE \n& \nCONTINUING EDUCATION \nB.COM II Semester 2023-24 \nStock Market Operation \n \nEDITED BY \nDr.M.NITHYA M.COM.,M.Phil.,M.B.A.,TN-SET.,Ph.D \nASSISTANT PROFESSOR (T) \nDEPARTMENT OF COMMERCE \nMANONMANIAM SUNDARANAR UNIVERISTY \nTIRUNELVELI - 12'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-03-07T11:57:24+00:00', 'source': '../SourceFile/JSCO21_IISem_StockMarketOperations.pdf', 'total_pages': 219, 'page': 1, 'page_label': '2'}, page_content='STOCK MARKET OPERATIONS \nSubject \nCode \n \nL \n \nT \n \nP \n \nS \n \nCredits Inst. \nHours \nMarks \nCIA External Total \n     2 2 25 75 100 \nLearning Objectives: \nLO1: To acquaint students with knowledge o

In [20]:
len(docs)

2518

In [21]:
len(wiki_documents)

76

In [34]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(new_doc, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [36]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002198E57A490>, search_kwargs={'k': 3})

In [46]:
prompt = ChatPromptTemplate.from_template(
    """
        Answer the Following question based on the provided context. 
        Think step by step before providing a detailed answer. 
        I will tip you $1000 if the user finds the answer useful.
        <context>
        {context}
        </context>
        Question : {input}                                          
    """
)

In [47]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(model,prompt)

In [48]:
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n        Answer the Following question based on the provided context. \n        Think step by step before providing a detailed answer. \n        I will tip you $1000 if the user finds the answer useful.\n        <context>\n        {context}\n        </context>\n        Question : {input}                                          \n    '), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000219CD28F230>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000219CD3116A0>, model_name='llama3-8b-81

In [49]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002198E57A490>, search_kwargs={'k': 3})

In [50]:
query = "What is Stockmarket and when was BSE established?"

retrival_chain = create_retrieval_chain(retriever,document_chain)
response = retrival_chain.invoke({"input": query}, config=RunnableConfig())

In [54]:
response['answer']

"I'm excited to answer your question!\n\nBased on the provided context, a stock market refers to a market where publicly traded companies issue and trade their shares of stock. It is a platform for buyers and sellers to trade securities, such as stocks, bonds, and other financial instruments.\n\nRegarding the Bombay Stock Exchange (BSE), it was established in 1875 as the Native Share and Stock Brokers' Association. It is also the first stock exchange in India and provides an equities trading platform for small-and-medium enterprises.\n\nSo, to summarize: a stock market is a platform for trading securities, and the Bombay Stock Exchange (BSE) was established in 1875 as the first stock exchange in India."

In [59]:
retrival_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002198E57A490>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n        Answer the Following question based on the provided context. \n        Think step by step before providing 